# Ch 33 (검증) — 작은 mask-diffusion LM 제대로 학습시키기

Ch 32 붕괴(유니그램 정체)를 교정한 레시피의 **실증 노트북**: vocab 2048 + 학습량 8000 step + carry-over 샘플러.

In [ ]:
%pip install -q -U transformers tokenizers datasets accelerate

In [ ]:
import math, time, torch
import torch.nn.functional as F
from datasets import load_dataset

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
print("torch", torch.__version__, "| device", device, "| fp16", USE_FP16)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. TinyStories 로드 (Ch 24/26과 동일 데이터)

In [ ]:
raw_train = load_dataset("roneneldan/TinyStories", split="train[:100000]")
raw_val   = load_dataset("roneneldan/TinyStories", split="validation[:500]")
print(raw_train)
print(raw_val[0]["text"][:160])

## 2. ★교정 1 — TinyStories에 BPE 2048 직접 학습 + `[MASK]` 추가

Ch 32는 `bert-base-uncased`(vocab 30522)를 가져와 임베딩이 파라미터의 ~70%를 차지했다. 작은 vocab으로 본체에 용량을 돌려준다.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast

VOCAB = 2048
def corpus_iter(bs=1000):
    for i in range(0, len(raw_train), bs):
        yield raw_train[i:i+bs]["text"]

_tk = Tokenizer(models.BPE(unk_token="[UNK]"))
_tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
_tk.decoder = decoders.ByteLevel()
_trainer = trainers.BpeTrainer(vocab_size=VOCAB, special_tokens=["[PAD]", "[UNK]", "[MASK]"])
_tk.train_from_iterator(corpus_iter(), trainer=_trainer)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=_tk, pad_token="[PAD]", unk_token="[UNK]", mask_token="[MASK]")
print("vocab_size :", tokenizer.vocab_size)
print("mask_id    :", tokenizer.mask_token_id, "| pad_id:", tokenizer.pad_token_id)
print("sample tok :", tokenizer.tokenize("Once upon a time there was a little cat.")[:14])

## 3. 토큰화 + group_texts (BLOCK_SIZE=128)

In [ ]:
BLOCK_SIZE = 128
def tok_fn(b):
    return tokenizer(b["text"], add_special_tokens=False)
tt = raw_train.map(tok_fn, batched=True, remove_columns=raw_train.column_names, desc="tok train")
tv = raw_val.map(tok_fn, batched=True, remove_columns=raw_val.column_names, desc="tok val")

def group_texts(b):
    cat = sum(b["input_ids"], [])
    n = (len(cat) // BLOCK_SIZE) * BLOCK_SIZE
    return {"input_ids": [cat[i:i+BLOCK_SIZE] for i in range(0, n, BLOCK_SIZE)]}
lm_train = tt.map(group_texts, batched=True, remove_columns=tt.column_names, desc="group train")
lm_val   = tv.map(group_texts, batched=True, remove_columns=tv.column_names, desc="group val")
print(f"train chunks {len(lm_train):,} | val {len(lm_val):,} | approx {len(lm_train)*BLOCK_SIZE/1e6:.2f}M tokens")

## 4. Diffusion collator — t~U(0.02,1) 하한 절단 + 재현성 generator (★교정 4)

In [ ]:
class DiffusionCollator:
    def __init__(self, tok, eps=0.02, seed=SEED):
        self.mask_id = tok.mask_token_id
        self.eps = eps
        self.gen = torch.Generator().manual_seed(seed)   # Trainer seed 와 분리
    def __call__(self, examples):
        ids = torch.tensor([e["input_ids"] for e in examples], dtype=torch.long)
        B, L = ids.shape
        t = torch.rand(B, generator=self.gen) * (1.0 - self.eps) + self.eps
        mask = torch.rand(B, L, generator=self.gen) < t.unsqueeze(1)
        no = ~mask.any(dim=1)
        if no.any():
            j = torch.randint(0, L, (int(no.sum()),), generator=self.gen)
            mask[no, j] = True
        inp = ids.clone(); inp[mask] = self.mask_id
        lab = ids.clone(); lab[~mask] = -100
        return {"input_ids": inp, "attention_mask": torch.ones(B, L, dtype=torch.long),
                "labels": lab, "t": t}
coll = DiffusionCollator(tokenizer)
print("collator ready, mask_id =", coll.mask_id)

## 5. 작은 BERT-MLM (본체 256/4L 유지 — ★검증서: 키우면 3h, 키우지 말 것)

In [ ]:
from transformers import BertConfig, BertForMaskedLM
cfg = BertConfig(vocab_size=tokenizer.vocab_size, hidden_size=256, num_hidden_layers=4,
                 num_attention_heads=4, intermediate_size=1024,
                 max_position_embeddings=BLOCK_SIZE, pad_token_id=tokenizer.pad_token_id)
model = BertForMaskedLM(cfg).to(device)
np_ = model.num_parameters()
emb = tokenizer.vocab_size * cfg.hidden_size
print(f"#params {np_/1e6:.2f}M | embedding share {emb/np_:.1%}  (Ch32: ~70%)")

## 6. ★교정 2 — diffusion loss(1/t)로 8000 step 학습

`loss = (마스크자리 CE 합 / L) / t` 의 배치평균. `/L`은 ELBO를 상수배한 surrogate(방향 동일). 1500=1.55분이었으니 8000도 예산 안.

In [ ]:
from transformers import Trainer, TrainingArguments

class DiffusionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        t = inputs["t"]; labels = inputs["labels"]
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        B, L, V = out.logits.shape
        per = F.cross_entropy(out.logits.view(-1, V), labels.view(-1),
                              ignore_index=-100, reduction="none").view(B, L)
        loss = ((per.sum(dim=1) / L) / t.to(per.dtype)).mean()
        return (loss, out) if return_outputs else loss

args = TrainingArguments(
    output_dir="./out33", max_steps=30000,
    per_device_train_batch_size=64, per_device_eval_batch_size=64,
    learning_rate=3e-4, weight_decay=0.01, warmup_steps=500,
    lr_scheduler_type="cosine", max_grad_norm=1.0, fp16=USE_FP16,
    logging_steps=250, eval_strategy="steps", eval_steps=2000, save_strategy="no",
    report_to="none", label_names=["labels"], remove_unused_columns=False, seed=SEED)

trainer = DiffusionTrainer(model=model, args=args, train_dataset=lm_train,
                           eval_dataset=lm_val, data_collator=coll)
t0 = time.time(); r = trainer.train(); el = (time.time()-t0)/60
print(f"\n=== summary ===\nelapsed {el:.2f} min | step {r.global_step} | train_loss {r.training_loss:.4f}")
print(f"random baseline ln(V) = {math.log(tokenizer.vocab_size):.4f}")
if torch.cuda.is_available():
    print(f"peak VRAM {torch.cuda.max_memory_allocated()/1024**2:.0f} MiB")

## 7. ★교정 3 — carry-over semi-AR 샘플러 (저신뢰 재마스킹 제거)

블록 단위로 왼→오 진행, 확정한 토큰은 불변(carry-over), 각 블록 마지막 step에 잔여 [MASK] 전부 확정. 생성에서만 mask_id 로짓 차단.

In [ ]:
@torch.no_grad()
def generate(model, length=128, block=32, temperature=0.8, top_p=0.92, top_k=0,
             rep_penalty=1.3, no_immediate_repeat=True, prompt_ids=None):
    """carry-over semi-AR + 반복 억제(rep penalty / 인접중복 금지 / top-p)."""
    model.eval()
    mask_id = tokenizer.mask_token_id
    x = torch.full((1, length), mask_id, dtype=torch.long, device=device)
    fixed = torch.zeros(length, dtype=torch.bool, device=device)
    if prompt_ids is not None:
        p = torch.tensor(prompt_ids[:length], device=device)
        x[0, :len(p)] = p; fixed[:len(p)] = True
    nblocks = (length + block - 1) // block
    for b in range(nblocks):
        lo, hi = b * block, min((b + 1) * block, length)
        steps = hi - lo
        for s in range(steps):
            logits = model(input_ids=x).logits[0].float()        # (L, V)
            logits[:, mask_id] = -1e9
            # 반복 패널티: 이미 확정된 토큰들의 로짓을 깎음
            if rep_penalty and rep_penalty != 1.0:
                comm = x[0][x[0] != mask_id]
                if comm.numel() > 0:
                    u = torch.unique(comm)
                    col = logits[:, u]
                    logits[:, u] = torch.where(col > 0, col / rep_penalty, col * rep_penalty)
            # 인접중복 금지: 각 자리에서 '왼쪽 토큰과 같은 토큰' 예측 차단
            if no_immediate_repeat:
                left = torch.roll(x[0], 1); left[0] = mask_id
                valid = left != mask_id
                logits[valid, left[valid]] = -1e9
            probs = (logits / max(temperature, 1e-6)).softmax(-1)
            if top_k and top_k > 0:
                kth = probs.topk(top_k, dim=-1).values[:, -1, None]
                probs = probs.masked_fill(probs < kth, 0.0)
            if top_p and top_p < 1.0:
                sp, si = probs.sort(dim=-1, descending=True)
                rm = (sp.cumsum(-1) - sp) > top_p
                sp = sp.masked_fill(rm, 0.0)
                probs = torch.zeros_like(probs).scatter(-1, si, sp)
            probs = probs / probs.sum(-1, keepdim=True).clamp_min(1e-9)
            pred = torch.multinomial(probs, 1).squeeze(-1)
            conf = probs.gather(-1, pred.unsqueeze(-1)).squeeze(-1)
            cur = (x[0] == mask_id) & (~fixed)
            cur[:lo] = False; cur[hi:] = False
            nleft = int(cur.sum())
            if nleft == 0: break
            nreveal = nleft if s == steps - 1 else max(1, nleft // (steps - s))
            cc = conf.clone(); cc[~cur] = -1e9
            idx = cc.topk(nreveal).indices
            x[0, idx] = pred[idx]
    return tokenizer.decode(x[0], skip_special_tokens=True)

print("=== 샘플러 sweep (같은 학습 모델, 조건부 'Once upon a time') ===")
pid = tokenizer("Once upon a time", add_special_tokens=False)["input_ids"]
configs = [
    ("A) 기존 temp0.7/topk40 (반복억제 없음)", dict(temperature=0.7, top_k=40, top_p=1.0, rep_penalty=1.0, no_immediate_repeat=False)),
    ("B) rep1.3 + 인접금지 + topp0.92",        dict(temperature=0.8, top_p=0.92, rep_penalty=1.3, no_immediate_repeat=True)),
    ("C) rep1.2 + temp0.9 + topp0.95",         dict(temperature=0.9, top_p=0.95, rep_penalty=1.2, no_immediate_repeat=True)),
    ("D) B + block16 (더 촘촘)",                dict(temperature=0.8, top_p=0.92, rep_penalty=1.3, no_immediate_repeat=True, block=16)),
]
torch.manual_seed(SEED)
for name, kw in configs:
    print(f"\n----- {name} -----")
    for i in range(2):
        print(f"[{i}] {generate(model, prompt_ids=pid, **kw)[:360]}")

## 8. 조건부 생성 + 정량 진단 (★교정 5: 고정-t top-1 acc / 유니그램 KL)

In [ ]:
# 정량 진단 (모델 자체 — 샘플러 무관)
g = torch.Generator().manual_seed(0)
def fixed_t_acc(t_val=0.15, n=128):
    cor = tot = 0
    for ex in lm_val.select(range(min(n, len(lm_val)))):
        ids = torch.tensor(ex["input_ids"])
        m = torch.rand(len(ids), generator=g) < t_val
        if not m.any(): m[0] = True
        inp = ids.clone(); inp[m] = tokenizer.mask_token_id
        with torch.no_grad():
            pr = model(inp.unsqueeze(0).to(device)).logits[0].argmax(-1).cpu()
        cor += (pr[m] == ids[m]).sum().item(); tot += int(m.sum())
    return cor / tot
print(f"[진단] 고정-t(0.15) top-1 accuracy = {fixed_t_acc():.3f}")

# 반복도 측정: 생성문의 4-gram 중복 비율 (낮을수록 좋음)
def rep4(text):
    toks = text.split()
    if len(toks) < 5: return 0.0
    grams = [tuple(toks[i:i+4]) for i in range(len(toks)-3)]
    return 1.0 - len(set(grams)) / len(grams)
import statistics
bestB = [generate(model, prompt_ids=pid, temperature=0.8, top_p=0.92, rep_penalty=1.3, no_immediate_repeat=True) for _ in range(5)]
baseA = [generate(model, prompt_ids=pid, temperature=0.7, top_k=40, top_p=1.0, rep_penalty=1.0, no_immediate_repeat=False) for _ in range(5)]
print(f"[진단] 4-gram 반복률  A(기존)={statistics.mean(map(rep4,baseA)):.3f}  B(반복억제)={statistics.mean(map(rep4,bestB)):.3f}  (낮을수록 좋음)")